## **Feature Engineering**

In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

## Load the Data from the data creation script

In [9]:
# 1. Search for the specific file starting from the parent directory
file_generator = Path.cwd().parent.rglob("transactions_2026.csv")

# 2. Get the path of the first match
target_file = next(file_generator)

# 3. Read it directly into Pandas
df = pd.read_csv(target_file)

## Feature engineering

In [10]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

### Define functions for generating new features

In [11]:
import numpy as np
import pandas as pd


# =======================================================================================
# Haversine distance (km)
# =======================================================================================
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate great-circle distance between two points on Earth.
    Inputs are in degrees. Output is in kilometers.
    """
    R = 6371.0  # Earth radius in km

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


# =======================================================================================
# Feature engineering
# =======================================================================================
def engineer_features(df):

    # =================================================================================
    # 0. Copy + enforce ordering (CRITICAL)
    # =================================================================================
    df = df.copy()
    df = df.sort_values(['user_id', 'timestamp'])

    
    
    # =================================================================================
    # 1. Time-based features
    # =================================================================================
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek

    
    
    # =================================================================================
    # 2. Transaction velocity (24h rolling count per user)
    # =================================================================================
    df['tx_count_24h'] = (
        df
        .groupby('user_id')
        .rolling('24h', on='timestamp')['tx_id']
        .count()
        .reset_index(level=0, drop=True)
    )

    # First transaction per user → count = 1
    df['tx_count_24h'] = df['tx_count_24h'].fillna(1)

    
    
    # =================================================================================
    # 3. Behavioral features
    # =================================================================================
    df['avg_spend_user'] = (
        df
        .groupby('user_id')['amount']
        .transform(lambda x: x.shift(1).expanding().mean())
    )

    df['amount_ratio'] = df['amount'] / (df['avg_spend_user'] + 1e-9)

    
    
    # =================================================================================
    # 4. Geospatial features
    # =================================================================================
    df['prev_lat'] = df.groupby('user_id')['lat'].shift(1)
    df['prev_lon'] = df.groupby('user_id')['lon'].shift(1)
    df['prev_ts']  = df.groupby('user_id')['timestamp'].shift(1)

    df['dist_from_last_tx_km'] = haversine(
        df['lat'], df['lon'],
        df['prev_lat'], df['prev_lon']
    ).fillna(0)

    time_diff_hours = (
        (df['timestamp'] - df['prev_ts'])
        .dt.total_seconds()
        .div(3600)
        .clip(lower=1e-3)
    )

    df['travel_velocity_kmph'] = (
        df['dist_from_last_tx_km'] / time_diff_hours
    ).fillna(0)

    
    
    # =================================================================================
    # 5. Categorical encoding
    # =================================================================================
    df = pd.get_dummies(
        df,
        columns=['auth_method', 'category'],
        drop_first=True
    )

    
    
    # =================================================================================
    # 6. Drop non-model columns
    # =================================================================================
    df = df.drop(
        columns=[
            'tx_id',
            'timestamp',
            'prev_lat',
            'prev_lon',
            'prev_ts'
        ]
    )

    
    
    # =================================================================================
    # 7. Final numeric safety
    # =================================================================================
    num_cols = df.select_dtypes(include='number').columns
    df[num_cols] = df[num_cols].fillna(0)

    return df

### Exporting the data with new features

In [12]:
# Run feature engineering
df_final = engineer_features(df)

# Save output
df_final.to_csv("fraud_features_ready.csv", index=False)

print("Features Engineered!")
print("Final shape:", df_final.shape)
print("Any NaNs left:", df_final.isna().sum())

ValueError: cannot reindex on an axis with duplicate labels